# SuSiEx with reference panel

In [1]:
import polars as pl
from pyprojroot.here import here

Prepare sumstats

In [2]:
# filter out null/unmatched variants
qc_mask = pl.any_horizontal(
    pl.all().is_null()
)

for dataset in ["mdd2024_afr", "mdd2024_eas", "mdd2024_his", "mdd2025_eur", "mdd2024_sas"]:
    sumstats = pl.scan_parquet(here(f"data/processed/sumstats/tidy/{dataset}/tidyGWAS_hivestyle"))
    susie = (sumstats
    .select(
        pl.col("CHR"),
        pl.col("RSID").alias("SNP"),
        pl.col("POS_37").alias("BP"),
        pl.col("EffectAllele").alias("A1"),
        pl.col("OtherAllele").alias("A2"),
        pl.col("B").alias("BETA"),
        pl.col("SE"),
        pl.col("P")
    )
    .filter(~qc_mask)
    )

    susie.sink_csv(here(f"data/processed/sumstats/{dataset}_hg19_susie.tsv"), separator = "\t")


Run SuSiEx

In [ ]:
%%bash -s {here()}
here=$1
sumstats=${here}/data/processed/sumstats
reference=${here}/data/processed/reference

sumstats_afr=${sumstats}/mdd2024_afr_hg19_susie.tsv
sumstats_eas=${sumstats}/mdd2024_eas_hg19_susie.tsv
sumstats_his=${sumstats}/mdd2024_his_hg19_susie.tsv
sumstats_sas=${sumstats}/mdd2024_sas_hg19_susie.tsv
sumstats_eur=${sumstats}/mdd2025_eur_hg19_susie.tsv 

ref_afr=${reference}/all_hg19_AFR_chr11_61000000-63000000
ref_eas=${reference}/all_hg19_EAS_chr11_61000000-63000000
ref_amr=${reference}/all_hg19_AMR_chr11_61000000-63000000
ref_sas=${reference}/all_hg19_SAS_chr11_61000000-63000000
ref_eur=${reference}/all_hg19_EUR_chr11_61000000-63000000

ld_afr=${reference}/ld/all_hg19_AFR_chr11_61000000-63000000
ld_eas=${reference}/ld/all_hg19_EAS_chr11_61000000-63000000
ld_amr=${reference}/ld/all_hg19_AMR_chr11_61000000-63000000
ld_sas=${reference}/ld/all_hg19_SAS_chr11_61000000-63000000
ld_eur=${reference}/ld/all_hg19_EUR_chr11_61000000-63000000

mkdir -p ${reference}/ld
mkdir -p ${here}/data/results/SuSiEx

$here/vendor/SuSiEx/bin_static/SuSiEx \
  --sst_file=$sumstats_afr,$sumstats_eas,$sumstats_his,$sumstats_sas,$sumstats_eur \
  --n_gwas=70727,58319,16211,14979,1577200 \
  --ref_file=$ref_afr,$ref_eas,$ref_amr,$ref_sas,$ref_eur \
  --ld_file=$ld_afr,$ld_eas,$ld_amr,$ld_sas,$ld_eur \
  --out_dir=${here}/data/results/SuSiEx \
  --out_name=mdd_all_hg19_chr11_61000000-63000000 \
  --chr=11 \
  --bp=61000000,63000000 \
  --chr_col=1,1,1,1,1 \
  --snp_col=2,2,2,2,2 \
  --bp_col=3,3,3,3,3 \
  --a1_col=4,4,4,4,4 \
  --a2_col=5,5,5,5,5 \
  --eff_col=6,6,6,6,6 \
  --se_col=7,7,7,7,7 \
  --pval_col=8,8,8,8,8 \
  --plink=${here}/.pixi/envs/default/bin/plink \
  --keep-ambig=True \
  --maf=0.005 \
  --mult-step=True \
  --pval_thresh=1e-5 \
  --max_iter=500 \
  --tol=1e-4 \
  --threads=8


Software parameters:
--sst_file = /home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_afr_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_eas_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_his_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_sas_hg19_susie.tsv,/home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2025_eur_hg19_susie.tsv
--ld_file = /home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_AFR_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_EAS_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_AMR_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_SAS_chr11_61000000-63000000,/home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_EUR_chr11_61000000-63000000
--n_gwas = 70

ax_iter = 500
--threads = 8
... parse reference file: /home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_AFR_chr11_61000000-63000000_ref.bim ...
... 12436 SNPs in the fine-mapping region read from /home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_AFR_chr11_61000000-63000000_ref.bim ...
... 9137 SNPs in the fine-mapping region read from /home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_afr_hg19_susie.tsv ...
... parse reference file: /home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_EAS_chr11_61000000-63000000_ref.bim ...
... 5825 SNPs in the fine-mapping region read from /home/madams23/Projects/cvd-mh-loci/data/processed/reference/ld/all_hg19_EAS_chr11_61000000-63000000_ref.bim ...
... 4130 SNPs in the fine-mapping region read from /home/madams23/Projects/cvd-mh-loci/data/processed/sumstats/mdd2024_eas_hg19_susie.tsv ...
... parse reference file: /home/madams23/Projects/cvd-mh-loci/data/processed/refe

Parse results

In [ ]:
cs = pl.read_csv(here("data/results/SuSiEx/mdd_all_hg19_chr11_61000000-63000000.cs"), separator="\t")
snp = pl.read_csv(here("data/results/SuSiEx/mdd_all_hg19_chr11_61000000-63000000.snp"), separator="\t")
summary = pl.read_csv(here("data/results/SuSiEx/mdd_all_hg19_chr11_61000000-63000000.summary"), separator="\t")

ComputeError: found more fields than defined in 'Schema'

Consider setting 'truncate_ragged_lines=True'.